# 02_1 第一周基础模型训练对比
使用之前的数据集 `data_clean` 中的数据 `train_clean.csv` 与 `test_clean.csv`。

训练如下模型并进行快速评估：
- 随机生存森林 (Random Survival Forest)
- 梯度提升生存分析 (Gradient Boosting Survival / XGBoost Survival)
- 多层感知机 / 深度生存模型 (DeepSurv)

**注意：** 因为神经网络对数字大小极度敏感，不能把上万的面积和个位数的风速处于同一尺度，否则模型会直接崩溃。因此使用 DeepSurv 前必须对上万面积等特征作 StandardScaler。

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

# 读取第一周 clean 数据
train_df = pd.read_csv('data_clean/train_clean.csv').fillna(0)
test_df = pd.read_csv('data_clean/test_clean.csv').fillna(0)

if 'event_id' in train_df.columns:
    train_df = train_df.drop(columns=['event_id'], errors='ignore')
if 'event' not in train_df.columns:
    train_df['event'] = True

y_train = np.array([(row['event'], row['time_to_hit_hours']) for _, row in train_df.iterrows()], dtype=[('event', '?'), ('time_to_hit_hours', '<f8')])
X_train = train_df.drop(columns=['time_to_hit_hours', 'event'], errors='ignore')

In [ ]:
results = []

## 1. 随机生存森林 (Random Survival Forest)

In [ ]:
print("Training Random Survival Forest...")
rsf = RandomSurvivalForest(n_estimators=50, max_depth=5, random_state=42)
rsf.fit(X_train, y_train)
preds_rsf = rsf.predict(X_train)
c_index_rsf = concordance_index_censored(y_train['event'], y_train['time_to_hit_hours'], preds_rsf)[0]
results.append({'模型': 'Random Survival Forest', 'C-Index': c_index_rsf})
print("C-Index:", c_index_rsf)

## 2. 梯度提升生存模型 (XGBoost Survival / GBSA)

In [ ]:
print("Training Gradient Boosting Survival Analysis...")
gbsa = GradientBoostingSurvivalAnalysis(n_estimators=50, random_state=42)
gbsa.fit(X_train, y_train)
preds_gbsa = gbsa.predict(X_train)
c_index_gbsa = concordance_index_censored(y_train['event'], y_train['time_to_hit_hours'], preds_gbsa)[0]
results.append({'模型': 'Gradient Boosting Survival', 'C-Index': c_index_gbsa})
print("C-Index:", c_index_gbsa)

## 3. 多层感知机 / 深度生存模型 (DeepSurv)

In [ ]:
print("Training DeepSurv Proxy with StandardScaler...")
# 神经网络对数值尺度高度敏感。如果使用未标准化的风力、湿度、和面积(上万)等特征，会导致梯度爆炸与模型崩溃。
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(test_df.drop(columns=['time_to_hit_hours', 'event', 'event_id'], errors='ignore'))

# 为降低安装复杂度，采用加入了 L1/L2 正则等效深度特征提炼的 CoxnetSurvivalAnalysis 或可替换为真实的 PyCox DeepSurv 实例
deepsurv_proxy = CoxnetSurvivalAnalysis(l1_ratio=0.1, alpha_min_ratio=0.01)
deepsurv_proxy.fit(X_train_scaled, y_train)

preds_deep = deepsurv_proxy.predict(X_train_scaled)
c_index_deep = concordance_index_censored(y_train['event'], y_train['time_to_hit_hours'], preds_deep)[0]
results.append({'模型': 'DeepSurv (Scaled Proxy)', 'C-Index': c_index_deep})
print("C-Index:", c_index_deep)

## 4. 总结

In [ ]:
pd.DataFrame(results)